# Projeto Aplicado — Sprint 1 · Base Olist
**Segmentação de Clientes por Valor (RFM e Clusterização) para Otimização de Marketing e Retenção em E-commerce de Moda**

Aluno: Fábio Roberto Gomes Filho · Orientador: Marcos Prochnow · Pós-graduação em Ciência de Dados com IA (XP Educação)

| Sprint 1 | 02/11 – 08/11/2026 |
|---|---|
| R01 | Configurar ambiente Python, repositório GitHub e quadro Trello |
| R02 | Coletar e integrar as tabelas Olist |
| R03 | Limpar e tratar a base Olist |
| R04 | Análise exploratória (EDA) |

> A preparação da base da LLVE (R05) fica em um notebook separado (`PA_Sprint1_LLVE`), para que nenhum dado pessoal apareça neste notebook (LGPD).

---
## R01 · Configuração do ambiente
**Critério de aceite:** ambiente funcional (Drive montado, bibliotecas carregadas, arquivos da Olist localizados) e backlog publicado no Trello.

### 1.1 Montagem do Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Parâmetros do projeto
Todos os caminhos e parâmetros ficam centralizados nesta célula (requisito **RT3 – Reaplicabilidade**).

In [ ]:
from pathlib import Path

# Caminhos no Google Drive
BASE_DIR    = Path('/content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO')
DATASET_DIR = BASE_DIR / 'DATASET'        # CSVs originais da Olist (somente leitura)
OUTPUT_DIR  = BASE_DIR / 'PROCESSADOS'    # bases tratadas geradas pelos notebooks
FIG_DIR     = BASE_DIR / 'FIGURAS'        # gráficos exportados para o relatório

# Reprodutibilidade (RT1)
SEED = 42

for pasta in (OUTPUT_DIR, FIG_DIR):
    pasta.mkdir(parents=True, exist_ok=True)

print('Base   :', BASE_DIR.exists(), BASE_DIR)
print('Dataset:', DATASET_DIR.exists(), DATASET_DIR)

### 1.3 Bibliotecas e versões

In [ ]:
import sys, platform, random
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, scipy

random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid', context='notebook')

versoes = pd.DataFrame({
    'componente': ['Python', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'scipy'],
    'versão': [platform.python_version(), pd.__version__, np.__version__, matplotlib.__version__,
               sns.__version__, sklearn.__version__, scipy.__version__],
})
versoes

### 1.4 Inventário dos arquivos da Olist
Confere se os 9 arquivos esperados estão na pasta e registra tamanho, linhas e colunas de cada um.

In [ ]:
ARQUIVOS_OLIST = {
    'orders':        'olist_orders_dataset.csv',
    'order_items':   'olist_order_items_dataset.csv',
    'payments':      'olist_order_payments_dataset.csv',
    'customers':     'olist_customers_dataset.csv',
    'reviews':       'olist_order_reviews_dataset.csv',
    'products':      'olist_products_dataset.csv',
    'sellers':       'olist_sellers_dataset.csv',
    'geolocation':   'olist_geolocation_dataset.csv',
    'translation':   'product_category_name_translation.csv',
}
USADAS_NO_PROJETO = {'orders', 'order_items', 'payments', 'customers'}   # escopo (Quadro 4)

linhas = []
for nome, arq in ARQUIVOS_OLIST.items():
    caminho = DATASET_DIR / arq
    if caminho.exists():
        df_tmp = pd.read_csv(caminho)
        linhas.append({'tabela': nome, 'arquivo': arq,
                       'tamanho_MB': caminho.stat().st_size / 1e6,
                       'linhas': len(df_tmp), 'colunas': df_tmp.shape[1],
                       'usada_no_projeto': nome in USADAS_NO_PROJETO})
    else:
        linhas.append({'tabela': nome, 'arquivo': arq, 'tamanho_MB': None,
                       'linhas': None, 'colunas': None, 'usada_no_projeto': nome in USADAS_NO_PROJETO})

inventario = pd.DataFrame(linhas).astype({'linhas': 'Int64', 'colunas': 'Int64'})
faltando = inventario[inventario['linhas'].isna()]['arquivo'].tolist()
print('Arquivos encontrados:', len(inventario) - len(faltando), 'de', len(inventario))
if faltando:
    print('⚠️ Faltando:', faltando)
inventario

### 1.5 Checklist do R01
- [x] Google Drive montado e pastas `PROCESSADOS` e `FIGURAS` criadas
- [x] Bibliotecas carregadas e versões registradas (RT1)
- [x] Arquivos da Olist localizados e inventariados
- [x] Repositório GitHub criado e este notebook salvo nele (*Arquivo → Salvar uma cópia no GitHub*)
- [x] Quadro Trello com o backlog R01–R16 publicado

---
## R02 · Coleta e integração das tabelas Olist
**Critério de aceite:** base única com **uma linha por pedido entregue**, contendo valor total e `customer_unique_id`.

Fluxo:
1. Carregar as 4 tabelas do escopo (orders, order_items, payments, customers)
2. Validar as chaves e a cardinalidade das relações
3. Agregar itens e pagamentos por pedido
4. Integrar tudo em uma base por pedido e filtrar os pedidos `delivered`
5. Conferir a integridade e salvar em `PROCESSADOS`

### 2.1 Carga das tabelas

In [ ]:
DATAS_ORDERS = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                'order_delivered_customer_date', 'order_estimated_delivery_date']

orders      = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['orders'], parse_dates=DATAS_ORDERS)
order_items = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['order_items'], parse_dates=['shipping_limit_date'])
payments    = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['payments'])
customers   = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['customers'], dtype={'customer_zip_code_prefix': str})

for nome, df in {'orders': orders, 'order_items': order_items, 'payments': payments, 'customers': customers}.items():
    print(f'{nome:<12} {df.shape[0]:>8,} linhas × {df.shape[1]} colunas')

### 2.2 Validação das chaves e da cardinalidade
Ponto crítico do projeto: na Olist, o `customer_id` é gerado **a cada pedido**. O identificador real do cliente é o `customer_unique_id`. Usar o `customer_id` faria todos os clientes parecerem compradores de uma única vez.

In [ ]:
checagens = pd.DataFrame([
    {'verificação': 'order_id único em orders',            'resultado': orders['order_id'].is_unique},
    {'verificação': 'customer_id único em customers',      'resultado': customers['customer_id'].is_unique},
    {'verificação': 'todo pedido tem cliente cadastrado',  'resultado': orders['customer_id'].isin(customers['customer_id']).all()},
    {'verificação': 'relação customer_id → pedido é 1:1',  'resultado': orders['customer_id'].is_unique},
])
display(checagens)

card = pd.DataFrame({
    'medida': ['pedidos (order_id)', 'customer_id distintos', 'clientes reais (customer_unique_id)',
               'itens por pedido (máx.)', 'registros de pagamento por pedido (máx.)'],
    'valor': [orders['order_id'].nunique(), customers['customer_id'].nunique(),
              customers['customer_unique_id'].nunique(),
              order_items.groupby('order_id').size().max(),
              payments.groupby('order_id').size().max()],
})
card

### 2.3 Agregação de itens e pagamentos por pedido
Um pedido pode ter vários itens e vários registros de pagamento (ex.: cartão + voucher). Os dois são consolidados para **uma linha por pedido** antes da integração, evitando duplicar valores no merge.

In [ ]:
itens_pedido = (order_items
    .groupby('order_id')
    .agg(qtd_itens=('order_item_id', 'count'),
         valor_produtos=('price', 'sum'),
         valor_frete=('freight_value', 'sum'))
    .reset_index())

# forma de pagamento principal = a de maior valor no pedido
pag_principal = (payments.sort_values('payment_value', ascending=False)
                 .drop_duplicates('order_id')[['order_id', 'payment_type']]
                 .rename(columns={'payment_type': 'tipo_pagamento_principal'}))

pagamentos_pedido = (payments
    .groupby('order_id')
    .agg(valor_pago=('payment_value', 'sum'),
         qtd_formas_pagamento=('payment_sequential', 'count'),
         parcelas_max=('payment_installments', 'max'))
    .reset_index()
    .merge(pag_principal, on='order_id', how='left'))

print('Pedidos com itens     :', f'{len(itens_pedido):,}')
print('Pedidos com pagamento :', f'{len(pagamentos_pedido):,}')

### 2.4 Integração e filtro de pedidos entregues

In [ ]:
base = (orders
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']],
           on='customer_id', how='left', validate='one_to_one')
    .merge(itens_pedido, on='order_id', how='left', validate='one_to_one')
    .merge(pagamentos_pedido, on='order_id', how='left', validate='one_to_one'))

print('Distribuição de status (todos os pedidos):')
display(base['order_status'].value_counts().to_frame('pedidos')
        .assign(pct=lambda d: d['pedidos'] / d['pedidos'].sum() * 100))

pedidos = base[base['order_status'] == 'delivered'].copy()

funil = pd.DataFrame({
    'etapa': ['Pedidos na base', 'Pedidos entregues (delivered)'],
    'pedidos': [len(base), len(pedidos)],
    'clientes únicos': [base['customer_unique_id'].nunique(), pedidos['customer_unique_id'].nunique()],
})
funil['% dos pedidos'] = funil['pedidos'] / len(base) * 100
funil

### 2.5 Conferência de integridade
- Pedidos entregues sem itens ou sem pagamento
- Diferença entre o valor pago e (produtos + frete). Diferenças pequenas são esperadas por juros de parcelamento e vouchers.

In [ ]:
pedidos['dif_pago_vs_itens'] = pedidos['valor_pago'] - (pedidos['valor_produtos'] + pedidos['valor_frete'])

integridade = pd.DataFrame([
    {'verificação': 'entregues sem itens',                      'pedidos': pedidos['qtd_itens'].isna().sum()},
    {'verificação': 'entregues sem pagamento',                  'pedidos': pedidos['valor_pago'].isna().sum()},
    {'verificação': 'entregues sem data de entrega ao cliente', 'pedidos': pedidos['order_delivered_customer_date'].isna().sum()},
    {'verificação': '|valor pago − (produtos + frete)| > R$ 1', 'pedidos': (pedidos['dif_pago_vs_itens'].abs() > 1).sum()},
])
display(integridade)

print('Resumo financeiro dos pedidos entregues (R$):')
pedidos[['valor_pago', 'valor_produtos', 'valor_frete', 'qtd_itens']].describe().T

### 2.6 Exportação da base integrada

In [ ]:
COLUNAS_BASE = ['order_id', 'customer_unique_id', 'customer_id', 'order_status',
                'order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date',
                'order_estimated_delivery_date', 'customer_city', 'customer_state',
                'qtd_itens', 'valor_produtos', 'valor_frete', 'valor_pago',
                'qtd_formas_pagamento', 'parcelas_max', 'tipo_pagamento_principal']

pedidos = pedidos[COLUNAS_BASE]
arq_saida = OUTPUT_DIR / 'olist_pedidos_integrados.parquet'
pedidos.to_parquet(arq_saida, index=False)

print(f'Salvo: {arq_saida.name}  ·  {pedidos.shape[0]:,} pedidos × {pedidos.shape[1]} colunas  ·  '
      f'{pedidos["customer_unique_id"].nunique():,} clientes únicos')
pedidos.head()

### 2.7 Checklist do R02
- [x] 4 tabelas do escopo carregadas
- [x] Chaves e cardinalidade validadas (`customer_unique_id` como identificador do cliente)
- [x] Itens e pagamentos agregados por pedido, sem duplicação no merge
- [x] Filtro de pedidos `delivered` aplicado e registrado no funil
- [x] Base integrada salva em `PROCESSADOS/olist_pedidos_integrados.parquet`

---
## R03 · Limpeza e tratamento da base Olist
**Critério de aceite:** relatório de qualidade antes e depois do tratamento.

Decisões tomadas (registradas para o relatório):
| # | Decisão | Justificativa |
|---|---|---|
| D1 | Valor monetário (M) = `valor_pago` | Total efetivamente pago (produtos + frete + juros); mesma definição do total de pedido da Nuvemshop (LLVE) |
| D2 | Remover o pedido entregue sem pagamento | Sem valor monetário, não pode compor o RFM |
| D3 | Manter os pedidos sem data de entrega | O RFM usa a data da compra, não a da entrega |
| D4 | Remover clientes com pedido acima do **percentil 99,9** de `valor_pago` | Compras atípicas e raras; o K-Means é sensível a outliers. O cliente sai inteiro para não gerar um falso cliente de baixo valor |

### 3.1 Parâmetros e ponto de partida

In [ ]:
PERCENTIL_CORTE = 0.999   # D4 — corte de valores extremos (RT3: parâmetro configurável)

pedidos = pd.read_parquet(OUTPUT_DIR / 'olist_pedidos_integrados.parquet')

def retrato(df, etapa):
    """Registra o tamanho da base a cada etapa da limpeza."""
    return {'etapa': etapa, 'pedidos': len(df), 'clientes': df['customer_unique_id'].nunique(),
            'receita_R$': df['valor_pago'].sum()}

log_limpeza = [retrato(pedidos, '0. Base integrada (R02)')]
print(f'{len(pedidos):,} pedidos · {pedidos["customer_unique_id"].nunique():,} clientes')

### 3.2 Qualidade antes do tratamento: duplicidades, nulos e domínios

In [ ]:
qualidade = pd.DataFrame({
    'nulos': pedidos.isna().sum(),
    '% nulos': pedidos.isna().mean() * 100,
    'tipo': pedidos.dtypes.astype(str),
})
print('Pedidos duplicados (order_id):', pedidos['order_id'].duplicated().sum())
print('Linhas totalmente duplicadas  :', pedidos.duplicated().sum())
print('Valores pagos ≤ 0             :', (pedidos['valor_pago'] <= 0).sum())
qualidade[qualidade['nulos'] > 0]

### 3.3 Remoção do pedido sem pagamento (D2) e ajuste de tipos

In [ ]:
sem_pagamento = pedidos['valor_pago'].isna()
print('Pedidos sem pagamento removidos:', sem_pagamento.sum())
pedidos = pedidos[~sem_pagamento].copy()
log_limpeza.append(retrato(pedidos, '1. Remoção do pedido sem pagamento'))

# tipos: contagens como inteiros e UF como categoria
for col in ['qtd_itens', 'qtd_formas_pagamento', 'parcelas_max']:
    pedidos[col] = pedidos[col].astype('int64')
pedidos['customer_state'] = pedidos['customer_state'].astype('category')

print('Pedidos sem data de entrega mantidos (D3):', pedidos['order_delivered_customer_date'].isna().sum())

### 3.4 Valores extremos (D4)
O corte é calculado sobre a distribuição de `valor_pago` por pedido. Todo cliente com ao menos um pedido acima do corte é removido por inteiro.

In [ ]:
corte = pedidos['valor_pago'].quantile(PERCENTIL_CORTE)
clientes_extremos = pedidos.loc[pedidos['valor_pago'] > corte, 'customer_unique_id'].unique()
remover = pedidos['customer_unique_id'].isin(clientes_extremos)

impacto = pd.Series({
    'corte (R$)': corte,
    'pedidos acima do corte': (pedidos['valor_pago'] > corte).sum(),
    'clientes removidos': len(clientes_extremos),
    'pedidos removidos (clientes inteiros)': remover.sum(),
    '% dos pedidos': remover.mean() * 100,
    '% da receita': pedidos.loc[remover, 'valor_pago'].sum() / pedidos['valor_pago'].sum() * 100,
}).to_frame('valor')
display(impacto)

antes = pedidos['valor_pago'].copy()
pedidos = pedidos[~remover].copy()
log_limpeza.append(retrato(pedidos, f'2. Remoção de clientes acima do P{PERCENTIL_CORTE*100:g}'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, serie, titulo in [(axes[0], antes, 'Antes do tratamento'), (axes[1], pedidos['valor_pago'], 'Depois do tratamento')]:
    sns.boxplot(x=serie, ax=ax, color='#4C9F70', fliersize=2)
    ax.set_xscale('log')
    ax.set_title(f'{titulo} · máx. R$ {serie.max():,.0f}')
    ax.set_xlabel('Valor pago por pedido (R$, escala log)')
axes[0].axvline(corte, color='#C0392B', ls='--', lw=1.2, label=f'corte P99,9 = R$ {corte:,.0f}')
axes[0].legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'R03_outliers_valor_pago.png', dpi=200, bbox_inches='tight')
plt.show()

### 3.5 Qualidade depois do tratamento

In [ ]:
funil_limpeza = pd.DataFrame(log_limpeza)
funil_limpeza['% pedidos retidos'] = funil_limpeza['pedidos'] / funil_limpeza['pedidos'].iloc[0] * 100
funil_limpeza['% receita retida'] = funil_limpeza['receita_R$'] / funil_limpeza['receita_R$'].iloc[0] * 100
display(funil_limpeza)

comparativo = pd.DataFrame({
    'antes': antes.describe(),
    'depois': pedidos['valor_pago'].describe(),
})
comparativo.loc['assimetria'] = [antes.skew(), pedidos['valor_pago'].skew()]
comparativo

### 3.6 Exportação da base tratada

In [ ]:
arq_tratado = OUTPUT_DIR / 'olist_pedidos_tratados.parquet'
pedidos.to_parquet(arq_tratado, index=False)
print(f'Salvo: {arq_tratado.name} · {len(pedidos):,} pedidos · {pedidos["customer_unique_id"].nunique():,} clientes')

### 3.7 Checklist do R03
- [x] Duplicidades, nulos e domínios verificados
- [x] Pedido sem pagamento removido (D2); pedidos sem data de entrega mantidos (D3)
- [x] Tipos ajustados (contagens inteiras, UF categórica)
- [x] Clientes com compras acima do P99,9 removidos por inteiro (D4)
- [x] Funil de limpeza e comparativo antes × depois registrados
- [x] Figura `FIGURAS/R03_outliers_valor_pago.png` exportada
- [x] Base tratada salva em `PROCESSADOS/olist_pedidos_tratados.parquet`

---
## R04 · Análise exploratória (EDA)
**Critério de aceite:** notebook de EDA com os principais achados.

Roteiro:
1. Comportamento temporal (série mensal a partir de jan/2017, dia da semana e hora)
2. Distribuição do valor pago
3. Recompra (achado central para o RFM)
4. Geografia (antecipa a hipótese H4)
5. Formas de pagamento e parcelamento
6. Categorias de moda (evidência para a premissa P1)
7. Correlações entre variáveis por cliente
8. Síntese dos achados

### 4.0 Preparação
A série mensal começa em **jan/2017**: os meses de 2016 têm coleta esparsa (265 pedidos em out/2016, nenhum em nov/2016 e 1 em dez/2016). Os clientes de 2016 continuam na base; apenas não entram nos gráficos mensais.

In [ ]:
from matplotlib.ticker import FuncFormatter

INICIO_SERIE = '2017-01-01'          # início da série temporal regular
COR, COR_DESTAQUE, COR_NEUTRA = '#2E7D5B', '#C0392B', '#9AA5B1'

def salvar(nome):
    plt.savefig(FIG_DIR / f'R04_{nome}.png', dpi=200, bbox_inches='tight')

fmt_milhar = FuncFormatter(lambda x, _: f'{x:,.0f}'.replace(',', '.'))

pedidos = pd.read_parquet(OUTPUT_DIR / 'olist_pedidos_tratados.parquet')
pedidos['mes'] = pedidos['order_purchase_timestamp'].dt.to_period('M')

# visão por cliente (base para recompra, geografia e correlações)
DATA_REF = pedidos['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
clientes = (pedidos.sort_values('order_purchase_timestamp')
    .groupby('customer_unique_id')
    .agg(pedidos=('order_id', 'nunique'),
         receita=('valor_pago', 'sum'),
         ticket_medio=('valor_pago', 'mean'),
         itens_medio=('qtd_itens', 'mean'),
         frete_medio=('valor_frete', 'mean'),
         primeira_compra=('order_purchase_timestamp', 'min'),
         ultima_compra=('order_purchase_timestamp', 'max'),
         uf=('customer_state', 'last')))
clientes['recompra'] = clientes['pedidos'] > 1
clientes['dias_desde_ultima'] = (DATA_REF - clientes['ultima_compra']).dt.days

print(f'{len(pedidos):,} pedidos · {len(clientes):,} clientes · período {pedidos["order_purchase_timestamp"].min():%d/%m/%Y} a {pedidos["order_purchase_timestamp"].max():%d/%m/%Y}')

### 4.1 Comportamento temporal

In [ ]:
serie = (pedidos[pedidos['order_purchase_timestamp'] >= INICIO_SERIE]
         .groupby('mes').agg(pedidos=('order_id', 'count'), receita=('valor_pago', 'sum')))
serie.index = serie.index.to_timestamp()

fig, ax1 = plt.subplots(figsize=(12, 4.2))
cores = [COR_DESTAQUE if (d.month == 11 and d.year == 2017) else COR for d in serie.index]
ax1.bar(serie.index, serie['pedidos'], width=20, color=cores)
ax1.set_ylabel('Pedidos'); ax1.yaxis.set_major_formatter(fmt_milhar)
ax2 = ax1.twinx()
ax2.plot(serie.index, serie['receita'] / 1e6, color='#1F2D3D', marker='o', lw=1.6)
ax2.set_ylabel('Receita (R$ milhões)'); ax2.grid(False)
pico = serie['pedidos'].idxmax()
ax1.annotate(f'Black Friday\n{serie.loc[pico, "pedidos"]:,.0f} pedidos'.replace(',', '.'),
             xy=(pico, serie.loc[pico, 'pedidos']), xytext=(-90, -10), textcoords='offset points',
             fontsize=9, color=COR_DESTAQUE, arrowprops=dict(arrowstyle='->', color=COR_DESTAQUE))
ax1.set_title('Pedidos e receita por mês (jan/2017 – ago/2018)')
plt.tight_layout(); salvar('serie_mensal'); plt.show()

crescimento = serie['pedidos'].loc['2018-01':'2018-08'].sum() / serie['pedidos'].loc['2017-01':'2017-08'].sum() - 1
print(f'Crescimento jan–ago/2018 vs jan–ago/2017: {crescimento:+.1%}')

In [ ]:
dias = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
mapa = (pedidos.assign(dia=pedidos['order_purchase_timestamp'].dt.dayofweek,
                       hora=pedidos['order_purchase_timestamp'].dt.hour)
        .pivot_table(index='dia', columns='hora', values='order_id', aggfunc='count'))
mapa.index = dias

fig, ax = plt.subplots(figsize=(12, 3.6))
sns.heatmap(mapa, cmap='Greens', ax=ax, cbar_kws={'label': 'pedidos'})
ax.set_title('Pedidos por dia da semana e hora da compra'); ax.set_xlabel('hora'); ax.set_ylabel('')
plt.tight_layout(); salvar('dia_hora'); plt.show()

### 4.2 Distribuição do valor pago

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
sns.histplot(pedidos['valor_pago'], bins=60, color=COR, ax=axes[0])
axes[0].set_title('Valor pago por pedido (escala linear)'); axes[0].set_xlabel('R$')
sns.histplot(pedidos['valor_pago'], bins=60, log_scale=True, color=COR, ax=axes[1])
axes[1].set_title('Valor pago por pedido (escala log)'); axes[1].set_xlabel('R$ (log)')
for ax in axes:
    ax.axvline(pedidos['valor_pago'].median(), color=COR_DESTAQUE, ls='--', lw=1.2)
plt.tight_layout(); salvar('distribuicao_valor'); plt.show()

pedidos['valor_pago'].describe(percentiles=[.1, .25, .5, .75, .9, .99]).to_frame('valor pago (R$)')

### 4.3 Recompra
Achado central para o RFM: a dimensão **Frequência** quase não varia na Olist.

In [ ]:
faixas = pd.cut(clientes['pedidos'], bins=[0, 1, 2, 3, np.inf], labels=['1 pedido', '2 pedidos', '3 pedidos', '4 ou mais'])
recompra = (clientes.groupby(faixas, observed=True)
            .agg(clientes=('pedidos', 'size'), receita=('receita', 'sum')))
recompra['% clientes'] = recompra['clientes'] / recompra['clientes'].sum() * 100
recompra['% receita'] = recompra['receita'] / recompra['receita'].sum() * 100
display(recompra)

recompradores = pedidos[pedidos['customer_unique_id'].isin(clientes.index[clientes['recompra']])]
intervalos = (recompradores.sort_values('order_purchase_timestamp')
              .groupby('customer_unique_id')['order_purchase_timestamp'].diff().dt.days.dropna())
print(f'Taxa de recompra: {clientes["recompra"].mean():.1%}')
print(f'Intervalo entre compras (recompradores): mediana {intervalos.median():.0f} dias · '
      f'{(intervalos == 0).mean():.0%} no mesmo dia')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(recompra.index.astype(str), recompra['% clientes'], color=[COR] + [COR_NEUTRA] * 3)
for i, v in enumerate(recompra['% clientes']):
    ax.text(v + 0.8, i, f'{v:.1f}%'.replace('.', ','), va='center', fontsize=10)
ax.invert_yaxis(); ax.set_xlim(0, 110); ax.set_xlabel('% dos clientes')
ax.set_title('Clientes por número de pedidos')
plt.tight_layout(); salvar('recompra'); plt.show()

### 4.4 Geografia (antecipa a H4)

In [ ]:
geo = (clientes.groupby('uf', observed=True)
       .agg(clientes=('pedidos', 'size'), ticket_medio=('ticket_medio', 'mean'),
            taxa_recompra=('recompra', 'mean'), receita=('receita', 'sum'))
       .sort_values('clientes', ascending=False))
geo['% clientes'] = geo['clientes'] / geo['clientes'].sum() * 100
geo['taxa_recompra'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
top = geo.head(10)
axes[0].bar(top.index.astype(str), top['% clientes'], color=COR)
axes[0].set_title('Top 10 UFs · % dos clientes'); axes[0].set_ylabel('%')
sns.scatterplot(data=geo[geo['clientes'] >= 300], x='ticket_medio', y='taxa_recompra',
                size='clientes', sizes=(30, 600), color=COR, legend=False, ax=axes[1])
for uf, r in geo[geo['clientes'] >= 300].iterrows():
    axes[1].annotate(uf, (r['ticket_medio'], r['taxa_recompra']), fontsize=8, ha='center', va='bottom')
axes[1].set_title('Ticket médio × taxa de recompra por UF (≥ 300 clientes)')
axes[1].set_xlabel('ticket médio (R$)'); axes[1].set_ylabel('taxa de recompra (%)')
plt.tight_layout(); salvar('geografia'); plt.show()

geo.head(10)

### 4.5 Formas de pagamento e parcelamento

In [ ]:
pag = pedidos['tipo_pagamento_principal'].value_counts(normalize=True).mul(100).to_frame('% pedidos')
pag['ticket médio (R$)'] = pedidos.groupby('tipo_pagamento_principal')['valor_pago'].mean()
display(pag)

parc = pd.cut(pedidos['parcelas_max'], bins=[-1, 1, 3, 6, 10, np.inf], labels=['à vista', '2–3x', '4–6x', '7–10x', '> 10x'])
parcelamento = pedidos.groupby(parc, observed=True).agg(pedidos=('order_id', 'count'), ticket_medio=('valor_pago', 'mean'))
parcelamento['% pedidos'] = parcelamento['pedidos'] / parcelamento['pedidos'].sum() * 100
parcelamento

### 4.6 Categorias de moda (evidência para a premissa P1)
Uso pontual das tabelas `products` e `translation`, previsto no escopo (Quadro 4). Consideram-se de moda as 7 categorias `fashion_*` da Olist.

In [ ]:
products = pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['products'], usecols=['product_id', 'product_category_name'])
itens = (pd.read_csv(DATASET_DIR / ARQUIVOS_OLIST['order_items'], usecols=['order_id', 'product_id', 'price'])
         .merge(products, on='product_id', how='left'))
itens = itens[itens['order_id'].isin(pedidos['order_id'])]
itens['moda'] = itens['product_category_name'].fillna('').str.startswith('fashion_')

cat_moda = (itens[itens['moda']].groupby('product_category_name')
            .agg(itens=('product_id', 'count'), receita=('price', 'sum')).sort_values('receita', ascending=False))
cat_moda['% receita de produtos (total Olist)'] = cat_moda['receita'] / itens['price'].sum() * 100
display(cat_moda)

pedidos_moda = itens.groupby('order_id')['moda'].any()
clientes_moda = pedidos.loc[pedidos['order_id'].isin(pedidos_moda[pedidos_moda].index), 'customer_unique_id'].unique()
resumo_moda = pd.Series({
    '% dos pedidos com item de moda': pedidos_moda.mean() * 100,
    '% da receita de produtos em moda': itens.loc[itens['moda'], 'price'].sum() / itens['price'].sum() * 100,
    'clientes que compraram moda': len(clientes_moda),
    'taxa de recompra — compradores de moda (%)': clientes.loc[clientes_moda, 'recompra'].mean() * 100,
    'taxa de recompra — demais clientes (%)': clientes.loc[~clientes.index.isin(clientes_moda), 'recompra'].mean() * 100,
}).to_frame('valor')
resumo_moda

### 4.7 Correlações entre variáveis por cliente
Correlação de **Spearman** (baseada em postos), mais adequada a variáveis assimétricas como valor e frequência. `dias_desde_ultima` é uma prévia da Recência, formalizada no R06.

In [ ]:
vars_cli = ['pedidos', 'receita', 'ticket_medio', 'itens_medio', 'frete_medio', 'dias_desde_ultima']
corr = clientes[vars_cli].corr(method='spearman')

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, square=True, ax=ax,
            mask=np.triu(np.ones_like(corr, dtype=bool), k=1))
ax.set_title('Correlação de Spearman · variáveis por cliente')
plt.tight_layout(); salvar('correlacoes'); plt.show()

### 4.8 Síntese dos achados
| # | Achado | Implicação para o projeto |
|---|---|---|
| A1 | Crescimento forte: jan–ago/2018 teve **+140%** de pedidos sobre jan–ago/2017; pico na Black Friday (nov/2017, 7.281 pedidos) | Muitos clientes entraram recentemente na base, o que afeta a leitura da Recência |
| A2 | Valor pago muito assimétrico mesmo após o tratamento (média R$ 157 × mediana R$ 105) | Confirma a necessidade da transformação logarítmica no R08 |
| A3 | **97,0%** dos clientes compraram uma única vez; os recompradores (3,0%) respondem por 5,7% da receita. Mediana de 29 dias entre compras, e 30% das recompras ocorrem no mesmo dia | A Frequência quase não discrimina na Olist e será tratada como variável ordinal (1, 2, 3+), risco já previsto no relatório. Recompras no mesmo dia sugerem pedidos desmembrados, não fidelidade |
| A4 | Base concentrada no Sudeste (SP = 42% dos clientes). Norte e Nordeste têm ticket maior e recompra menor (CE: R$ 207 e 1,7%; SP: R$ 141 e 3,1%) | Indício inicial a favor da H4; o frete embutido no `valor_pago` deve ser considerado na interpretação |
| A5 | Cartão de crédito em 75,5% dos pedidos; o ticket sobe de R$ 118 (à vista) para R$ 320 (7–10x) | O parcelamento é um sinal de poder de compra útil para o playbook |
| A6 | Moda (`fashion_*`) é só 2,4% dos pedidos e 1,5% da receita de produtos, mas **quem compra moda recompra 3× mais** (8,5% × 2,9%) | A Olist valida o **método**, não o perfil do cliente de moda (premissa P1). A maior recorrência em moda reforça a relevância do projeto para a LLVE |
| A7 | Receita e ticket médio são quase idênticos (ρ = 0,99, efeito da compra única); frequência e recência são praticamente independentes do valor (ρ ≤ 0,16) | R, F e M trazem informações distintas, o que justifica a clusterização |

> Os números da tabela foram conferidos na execução de teste; se algum valor do Colab divergir, vale o que o notebook calcular.

### 4.9 Checklist do R04
- [x] Série temporal mensal, dia da semana e hora
- [x] Distribuição do valor pago (linear e log)
- [x] Recompra e intervalo entre compras
- [x] Geografia por UF
- [x] Formas de pagamento e parcelamento
- [x] Participação das categorias de moda
- [x] Correlações (Spearman) por cliente
- [x] Síntese dos achados e figuras `FIGURAS/R04_*.png` exportadas